# Manafwa Flood Early Warning — Threshold Engineering + XGBoost Risk Classifier

**Pipeline**
1. Pull long-run historical GloFAS discharge (Open-Meteo Flood API, seamless model, 1984→today) for the Manafwa catchment
2. Engineer lag / rolling / seasonal features and derive percentile-based risk thresholds (Normal / Mild / Advanced / Extreme)
3. Train an XGBoost multiclass classifier on the engineered history
4. Pull the live 7-day ensemble forecast and score it with the trained model
5. Combine **four independent criteria** into a single alert decision, so a single noisy value can't trigger a false red alert:
   - Model confidence ≥ 85% **and** predicted class == Extreme
   - Forecast ensemble **maximum** discharge crosses the Extreme threshold
   - Ensemble **member agreement** (majority of the 50 GloFAS members must agree)
   - **Persistence**: the flood signal holds across ≥2 consecutive forecast pulls / lead times, not just one

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score, roc_auc_score)
from sklearn.preprocessing import label_binarize

import xgboost as xgb

# pip install imbalanced-learn if not already available
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (11, 4)

RANDOM_STATE = 42

In [ ]:
# River Manafwa catchment
MANAFWA_LAT = 0.9333
MANAFWA_LON = 34.3667

FLOOD_API_BASE = "https://flood-api.open-meteo.com/v1/flood"

# Risk classes — kept consistent with the thesis encoding
CLASS_NAMES = {0: "Normal", 1: "Mild", 2: "Advanced", 3: "Extreme"}
EXTREME_CLASS = 3


## 2. Pull long-run historical discharge

In [ ]:
def fetch_historical_discharge(lat, lon, start_date="2000-01-01", end_date=None,
                                cell_selection="land"):
    end_date = end_date or date.today().isoformat()
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "river_discharge",
        "start_date": start_date,
        "end_date": end_date,
        "cell_selection": cell_selection,
        "timeformat": "iso8601",
    }
    r = requests.get(FLOOD_API_BASE, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "daily" not in data:
        raise ValueError(f"Unexpected response: {data}")
    df = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "discharge": data["daily"]["river_discharge"],
    })
    return df.dropna().reset_index(drop=True)

hist_df = fetch_historical_discharge(MANAFWA_LAT, MANAFWA_LON)
print(hist_df.shape)
hist_df.head()


In [ ]:
hist_df.plot(x="date", y="discharge", title="GloFAS discharge — Manafwa (raw history)")
plt.ylabel("Discharge (m³/s)")
plt.show()


**Optional — bias check against your MWE gauge data.** If you have your thesis's cleaned
1988–2025 MWE discharge series loaded as `mwe_df` (columns `date`, `discharge`), merge on date
and eyeball the ratio. If it's consistently offset by a roughly constant factor, you can apply a
simple ratio or quantile-mapping correction before thresholding — otherwise the self-referential
percentile approach below still works because it only compares GloFAS to itself.

```python
# merged = hist_df.merge(mwe_df, on="date", suffixes=("_glofas", "_mwe"))
# merged["ratio"] = merged["discharge_glofas"] / merged["discharge_mwe"]
# merged["ratio"].describe()
```


## 3. Feature engineering

In [ ]:
def engineer_features(df):
    df = df.sort_values("date").reset_index(drop=True).copy()

    # Lag features
    for lag in [1, 2, 3, 5, 7, 14]:
        df[f"discharge_lag{lag}"] = df["discharge"].shift(lag)

    # Rolling stats (shifted by 1 so we never leak the current day into itself)
    for window in [3, 7, 14, 30]:
        roll = df["discharge"].shift(1).rolling(window)
        df[f"discharge_roll_mean{window}"] = roll.mean()
        df[f"discharge_roll_std{window}"] = roll.std()
        df[f"discharge_roll_max{window}"] = roll.max()

    # Rate of change
    df["discharge_pct_change1"] = df["discharge"].pct_change(1)
    df["discharge_pct_change3"] = df["discharge"].pct_change(3)
    df["discharge_diff1"] = df["discharge"].diff(1)

    # Seasonality (Uganda: two rainy seasons roughly Mar-May and Aug-Nov)
    df["month"] = df["date"].dt.month
    df["day_of_year"] = df["date"].dt.dayofyear
    df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["is_rainy_season"] = df["month"].isin([3, 4, 5, 8, 9, 10, 11]).astype(int)

    return df

feat_df = engineer_features(hist_df)
feat_df.tail()


## 4. Percentile-based thresholds and class labels

Thresholds are computed **per calendar month** rather than on the pooled series, since "extreme"
in the dry season and "extreme" in a rainy month are not the same absolute discharge. This
mirrors the quartile-based approach from the thesis (Normal/Mild/Advanced/Extreme, 0–3) but adds
seasonal awareness, which is more defensible for issuing real-time alerts.


In [ ]:
def build_monthly_thresholds(df, q_mild=0.50, q_advanced=0.75, q_extreme=0.90):
    """Returns a DataFrame indexed by month with the 3 cut points."""
    thresholds = (
        df.groupby(df["date"].dt.month)["discharge"]
        .quantile([q_mild, q_advanced, q_extreme])
        .unstack()
    )
    thresholds.columns = ["p_mild", "p_advanced", "p_extreme"]
    thresholds.index.name = "month"
    return thresholds

monthly_thresholds = build_monthly_thresholds(hist_df)
monthly_thresholds


In [ ]:
def label_risk_class(df, thresholds):
    df = df.copy()
    t = df["month"].map(thresholds["p_mild"])
    a = df["month"].map(thresholds["p_advanced"])
    e = df["month"].map(thresholds["p_extreme"])

    conditions = [df["discharge"] >= e, df["discharge"] >= a, df["discharge"] >= t]
    choices = [3, 2, 1]
    df["risk_class"] = np.select(conditions, choices, default=0)
    return df

feat_df = label_risk_class(feat_df, monthly_thresholds)
feat_df["risk_class"].value_counts().sort_index().rename(index=CLASS_NAMES)


In [ ]:
feat_df["risk_class"].value_counts(normalize=True).sort_index().rename(index=CLASS_NAMES)


## 5. Train / test split, SMOTE, XGBoost

Chronological 80/20 split and SMOTE applied to minority classes 

In [ ]:
FEATURE_COLS = [c for c in feat_df.columns
                if c not in ["date", "discharge", "risk_class", "month", "day_of_year"]]

model_df = feat_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

X_train, y_train = train_df[FEATURE_COLS], train_df["risk_class"]
X_test, y_test = test_df[FEATURE_COLS], test_df["risk_class"]

print("Train:", X_train.shape, " Test:", X_test.shape)
y_train.value_counts().sort_index()


In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
y_train_res.value_counts().sort_index()


In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
)

xgb_model.fit(X_train_res, y_train_res)



In [ ]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
extreme_f1 = f1_score(y_test == EXTREME_CLASS, y_pred == EXTREME_CLASS)

print(f"Accuracy:        {acc:.3f}")
print(f"Macro F1:        {macro_f1:.3f}")
print(f"Extreme-class F1:{extreme_f1:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=[CLASS_NAMES[i] for i in range(4)]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES.values(), rotation=45)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES.values())
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Confusion matrix — risk class")
plt.colorbar(im)
plt.tight_layout()
plt.show()


## 6. Live 7-day ensemble forecast

In [ ]:
def fetch_forecast_discharge(lat, lon, forecast_days=7, cell_selection="land"):
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "river_discharge,river_discharge_mean,river_discharge_median,"
                 "river_discharge_max,river_discharge_min,river_discharge_p25,river_discharge_p75",
        "forecast_days": forecast_days,
        "cell_selection": cell_selection,
        "timeformat": "iso8601",
    }
    r = requests.get(FLOOD_API_BASE, params=params, timeout=60)
    r.raise_for_status()
    return pd.DataFrame(r.json()["daily"]).assign(date=lambda d: pd.to_datetime(d["time"]))


def fetch_forecast_ensemble(lat, lon, forecast_days=7, cell_selection="land"):
    """All 50 GloFAS members, for the member-agreement criterion."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "river_discharge",
        "forecast_days": forecast_days,
        "ensemble": "true",
        "cell_selection": cell_selection,
        "timeformat": "iso8601",
    }
    r = requests.get(FLOOD_API_BASE, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()["daily"]
    dates = pd.to_datetime(data.pop("time"))
    member_cols = {k: v for k, v in data.items() if k.startswith("river_discharge")}
    ens_df = pd.DataFrame(member_cols, index=dates)
    return ens_df

forecast_df = fetch_forecast_discharge(MANAFWA_LAT, MANAFWA_LON)
ensemble_df = fetch_forecast_ensemble(MANAFWA_LAT, MANAFWA_LON)

display(forecast_df)
display(ensemble_df.head())


`ensemble=true` returns one column per GloFAS member (typically `river_discharge_member01` …
`member50`, naming can vary slightly — check `ensemble_df.columns` after your first live call
and adjust the member-agreement function below if needed).


## 7. Score the forecast with the trained model

To compute lag/rolling features for the forecast days we need continuity with recent history —
we stitch the last 30 days of historical discharge onto the forecast series, re-run the same
`engineer_features` function, then take only the forecast rows back out.


In [ ]:
def build_forecast_features(hist_df, forecast_df, thresholds):
    recent_hist = hist_df[["date", "discharge"]].tail(30)
    fcst = forecast_df[["date", "river_discharge"]].rename(columns={"river_discharge": "discharge"})
    combined = pd.concat([recent_hist, fcst], ignore_index=True).drop_duplicates("date")
    combined = engineer_features(combined)
    combined = label_risk_class(combined, thresholds)  # not used for training, just for reference
    forecast_feat = combined[combined["date"].isin(fcst["date"])].reset_index(drop=True)
    return forecast_feat

forecast_feat = build_forecast_features(hist_df, forecast_df, monthly_thresholds)
X_forecast = forecast_feat[FEATURE_COLS]

forecast_pred = xgb_model.predict(X_forecast)
forecast_proba = xgb_model.predict_proba(X_forecast)
forecast_confidence = forecast_proba.max(axis=1)

forecast_feat["predicted_class"] = forecast_pred
forecast_feat["predicted_label"] = forecast_feat["predicted_class"].map(CLASS_NAMES)
forecast_feat["confidence"] = forecast_confidence

forecast_feat[["date", "discharge", "predicted_label", "confidence"]]


## 8. Combine criteria into a single alert decision

Four independent checks, each of which can be individually false-alarm-prone on its own, are
combined so a single noisy signal can't flip the alert to red — but requiring **all four**
turned out to be too strict once validated against real events (see Section 13): it lets one
weak or biased criterion veto the other three. **3-of-4 is the Red gate below**, not 4-of-4.

| # | Criterion | What it catches |
|---|-----------|------------------|
| 1 | XGBoost predicted class == Extreme **and** confidence ≥ 85% | Model-based pattern recognition |
| 2 | Forecast **maximum** discharge (`river_discharge_max`) crosses the month's Extreme threshold | Worst-case ensemble scenario |
| 3 | **Member agreement** ≥ 50% of the 50 GloFAS members cross the Extreme threshold | Rules out a single outlier member driving the alert |
| 4 | **Persistence** — the same day keeps flagging across ≥2 consecutive forecast pulls (or across the 5-day → 3-day → day-of lead-time sequence) | Rules out a one-off noisy forecast run |

Alert color:
- 🟢 **Green** — 0–1 criteria met
- 🟠 **Orange** — 2 criteria met (elevated, monitor closely, prep messaging)
- 🔴 **Red** — 3 or 4 criteria met (issue the SMS/GPS push)


In [ ]:
def member_agreement_flag(ensemble_row, extreme_threshold, min_agreement=0.5):
    members = ensemble_row.dropna()
    if len(members) == 0:
        return False, 0.0
    frac_exceeding = (members >= extreme_threshold).mean()
    return frac_exceeding >= min_agreement, frac_exceeding


def evaluate_alert_criteria(forecast_feat, forecast_df, ensemble_df, thresholds,
                             previous_run_flags=None, min_confidence=0.85,
                             min_member_agreement=0.5, red_gate=3):
    results = []
    current_run_flags = {}

    for _, row in forecast_feat.iterrows():
        d = row["date"]
        month = d.month
        extreme_thr = thresholds.loc[month, "p_extreme"]

        # Criterion 1 — model
        crit_model = (row["predicted_class"] == EXTREME_CLASS) and (row["confidence"] >= min_confidence)

        # Criterion 2 — forecast max vs threshold
        fcst_row = forecast_df.loc[forecast_df["date"] == d]
        max_discharge = fcst_row["river_discharge_max"].values[0] if len(fcst_row) else np.nan
        crit_max = bool(max_discharge >= extreme_thr) if not np.isnan(max_discharge) else False

        # Criterion 3 — ensemble member agreement
        crit_agree, agree_frac = False, 0.0
        if d in ensemble_df.index:
            crit_agree, agree_frac = member_agreement_flag(
                ensemble_df.loc[d], extreme_thr, min_member_agreement)

        # Criterion 4 — persistence across runs
        day_key = d.strftime("%Y-%m-%d")
        flagged_now = crit_model or crit_max or crit_agree
        current_run_flags[day_key] = flagged_now
        crit_persist = bool(previous_run_flags and previous_run_flags.get(day_key, False) and flagged_now)

        n_met = sum([crit_model, crit_max, crit_agree, crit_persist])
        if n_met >= red_gate:
            color = "RED"
        elif n_met >= 2:
            color = "ORANGE"
        else:
            color = "GREEN"

        results.append({
            "date": day_key,
            "discharge_forecast": row["discharge"],
            "predicted_label": row["predicted_label"],
            "model_confidence": round(row["confidence"], 3),
            "extreme_threshold": round(extreme_thr, 2),
            "crit_model": crit_model,
            "crit_forecast_max": crit_max,
            "crit_member_agreement": crit_agree,
            "member_agreement_frac": round(agree_frac, 2),
            "crit_persistence": crit_persist,
            "criteria_met": n_met,
            "alert_color": color,
        })

    return pd.DataFrame(results), current_run_flags


alert_table, run_flags = evaluate_alert_criteria(
    forecast_feat, forecast_df, ensemble_df, monthly_thresholds,
    previous_run_flags=None,   # load from disk / DB on subsequent runs — see Section 9
)
alert_table


## 9. Persisting `run_flags` between pipeline runs

The persistence criterion needs to know what the *previous* forecast run said about the same
calendar day. In production, run this pipeline on a schedule (e.g. every 6 hours) and persist
`run_flags` (a small JSON: `{date: bool}`) to disk, a database row, or your app's backend between
runs — then pass it back in as `previous_run_flags` on the next call. A minimal file-based version:


In [ ]:
import json as _json
from pathlib import Path

FLAGS_PATH = Path("last_run_flags.json")

def load_previous_flags():
    if FLAGS_PATH.exists():
        return _json.loads(FLAGS_PATH.read_text())
    return None

def save_current_flags(flags):
    FLAGS_PATH.write_text(_json.dumps(flags))

# Example scheduled-run pattern:
# previous = load_previous_flags()
# alert_table, current = evaluate_alert_criteria(forecast_feat, forecast_df, ensemble_df,
#                                                 monthly_thresholds, previous_run_flags=previous)
# save_current_flags(current)


## 10. Mapping alerts to your 5-day / 3-day / day-of / hours-out schedule

The Flood API is **daily-resolution** and gives **most recent same-day pull** — if you re-run this
   pipeline every few hours on the day of the flood.

In [ ]:
today = pd.Timestamp(date.today())
alert_table["date_dt"] = pd.to_datetime(alert_table["date"])
alert_table["lead_days"] = (alert_table["date_dt"] - today).dt.days

for lead, label in [(5, "5-day advisory"), (3, "3-day advisory"), (0, "Day-of alert")]:
    row = alert_table.loc[alert_table["lead_days"] == lead]
    if not row.empty:
        r = row.iloc[0]
        print(f"[{label}] {r['date']} — {r['alert_color']} "
              f"({r['criteria_met']}/4 criteria, {r['predicted_label']}, "
              f"confidence={r['model_confidence']})")


## 12. Scan-based case study — find days where the criteria were actually met

Rather than checking a date you already suspect, scan the **entire engineered history** and let
the criteria themselves surface qualifying days. This is stronger evidence for a demo (`"the
logic independently found N historical Red-alert days"`) and may also surface events your thesis
didn't explicitly flag.

Note on criterion 4 here: same limitation as Section 11 — no ensemble members exist in the
historical reanalysis, so this scan uses the same **flood-signature proxy** in place of member
agreement: a *sustained* signature (smooth, multi-day high flow — captures broad regional
anomalies like 1997/98) OR'd with an *onset* signature (a sharp single-day jump — captures
landslide-triggered flash floods, this catchment's more common pattern per the ReliefWeb record
in Section 13). Treat the results as candidate days for manual review against known flood
records, not a final verified event list.


In [ ]:
def scan_history_for_alerts(feat_df, model, thresholds, min_confidence=0.85, window=3,
                             onset_pct_jump=0.5, red_gate=3):
    df = feat_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    proba = model.predict_proba(df[FEATURE_COLS])
    df["predicted_class"] = proba.argmax(axis=1)
    df["confidence"] = proba.max(axis=1)

    extreme_thr_by_row = df["month"].map(thresholds["p_extreme"])
    df["crit_model"] = (df["predicted_class"] == EXTREME_CLASS) & (df["confidence"] >= min_confidence)
    df["crit_threshold"] = df["discharge"] >= extreme_thr_by_row

    prev_class = df["predicted_class"].shift(1)
    df["crit_persistence"] = (df["predicted_class"] == EXTREME_CLASS) & (prev_class == EXTREME_CLASS)

    roll = df["discharge"].rolling(window)
    cv = roll.std() / roll.mean()
    crit_sustained = (roll.mean() >= extreme_thr_by_row) & (cv < 0.25)

    pct_jump = df["discharge_pct_change1"]
    crit_onset = (df["discharge"] >= extreme_thr_by_row) & (pct_jump >= onset_pct_jump)

    df["crit_flood_signature"] = crit_sustained.fillna(False) | crit_onset.fillna(False)

    df["criteria_met"] = df[["crit_model", "crit_threshold", "crit_persistence",
                              "crit_flood_signature"]].sum(axis=1)
    df["alert_color"] = np.select(
        [df["criteria_met"] >= red_gate, df["criteria_met"] >= 2],
        ["RED", "ORANGE"], default="GREEN")
    return df

scan_df = scan_history_for_alerts(feat_df, xgb_model, monthly_thresholds)

red_days = scan_df[scan_df["alert_color"] == "RED"][
    ["date", "discharge", "risk_class", "confidence", "criteria_met"]
].reset_index(drop=True)
red_days["risk_class"] = red_days["risk_class"].map(CLASS_NAMES)

print(f"RED days found across full history: {len(red_days)}")
print(f"ORANGE days found: {(scan_df['alert_color'] == 'ORANGE').sum()}")
red_days.head(20)


`scan_df["alert_color"]` gives every day a label — so besides listing individual Red days you can
also check the overall base rate (what fraction of days would have fired Red/Orange), which is a
useful sanity number: too high and your team will get alert fatigue, too low and you're probably
missing real events.


In [ ]:
scan_df["alert_color"].value_counts(normalize=True).rename("share_of_days")


**Group consecutive Red days into discrete episodes** — a 3-day Red stretch is one flood episode,
not three separate case studies.


In [ ]:
def group_into_episodes(scan_df, color="RED", gap_days=3):
    hits = scan_df[scan_df["alert_color"] == color].sort_values("date").reset_index(drop=True)
    if hits.empty:
        return pd.DataFrame(columns=["start", "end", "peak_discharge", "n_days"])

    episodes = []
    start = hits.loc[0, "date"]
    prev = start
    for d in hits["date"].iloc[1:]:
        if (d - prev).days > gap_days:
            episodes.append((start, prev))
            start = d
        prev = d
    episodes.append((start, prev))

    rows = []
    for s, e in episodes:
        window = scan_df[(scan_df["date"] >= s) & (scan_df["date"] <= e)]
        rows.append({
            "start": s.strftime("%Y-%m-%d"),
            "end": e.strftime("%Y-%m-%d"),
            "peak_discharge": round(window["discharge"].max(), 1),
            "n_days": len(window),
        })
    return pd.DataFrame(rows)

red_episodes = group_into_episodes(scan_df, color="RED")
print(f"Discrete RED episodes: {len(red_episodes)}")
red_episodes


Pick one episode from `red_episodes` and inspect it in detail with the same window-plot used in
Section 11 — just pass its start date in as the "event" center. This works for *any* Red day the
scan found, known flood or not.


In [ ]:
def backtest_event(feat_df, model, thresholds, event_date, days_before=7, days_after=2,
                    min_confidence=0.85, window=3, onset_pct_jump=0.5, red_gate=3):
   
    event_date = pd.Timestamp(event_date)
    win_df = feat_df[
        (feat_df["date"] >= event_date - pd.Timedelta(days=days_before))
        & (feat_df["date"] <= event_date + pd.Timedelta(days=days_after))
    ].copy().reset_index(drop=True)

    win_df = win_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    if win_df.empty:
        raise ValueError(f"No usable feature rows around {event_date.date()} "
                          f"-- check the date falls within hist_df's range and has no gaps.")

    proba = model.predict_proba(win_df[FEATURE_COLS])
    win_df["predicted_class"] = proba.argmax(axis=1)
    win_df["predicted_label"] = win_df["predicted_class"].map(CLASS_NAMES)
    win_df["confidence"] = proba.max(axis=1)

    rows = []
    for i, row in win_df.iterrows():
        month = row["date"].month
        extreme_thr = thresholds.loc[month, "p_extreme"]

        crit_model = (row["predicted_class"] == EXTREME_CLASS) and (row["confidence"] >= min_confidence)
        crit_threshold = bool(row["discharge"] >= extreme_thr)

        # Persistence proxy: was Extreme also predicted on the previous day in this window?
        crit_persist = False
        if i > 0:
            prev = win_df.iloc[i - 1]
            crit_persist = prev["predicted_class"] == EXTREME_CLASS and row["predicted_class"] == EXTREME_CLASS

        # Criterion 4 (BACKTEST-ONLY proxy for ensemble member agreement): OR of two flood
        # signatures, since a single "smooth" definition misses spiky, landslide-triggered
        # flash floods, which are this catchment's dominant pattern.
        lo = max(0, i - window + 1)
        recent = win_df.iloc[lo:i + 1]["discharge"]
        cv = recent.std() / recent.mean() if recent.mean() else np.nan
        crit_sustained = bool(recent.mean() >= extreme_thr and (np.isnan(cv) or cv < 0.25))

        pct_jump = row.get("discharge_pct_change1", np.nan)
        crit_onset = bool(row["discharge"] >= extreme_thr and pd.notna(pct_jump) and pct_jump >= onset_pct_jump)

        crit_flood_signature = crit_sustained or crit_onset

        n_met = sum([crit_model, crit_threshold, crit_persist, crit_flood_signature])
        color = "RED" if n_met >= red_gate else ("ORANGE" if n_met >= 2 else "GREEN")

        rows.append({
            "date": row["date"].strftime("%Y-%m-%d"),
            "discharge": round(row["discharge"], 1),
            "actual_risk_class": CLASS_NAMES.get(row.get("risk_class", np.nan), "n/a"),
            "predicted_label": row["predicted_label"],
            "confidence": round(row["confidence"], 3),
            "crit_model": crit_model,
            "crit_threshold": crit_threshold,
            "crit_persistence": crit_persist,
            "crit_sustained(*)": crit_sustained,
            "crit_onset(*)": crit_onset,
            "criteria_met": n_met,
            "alert_color": color,
            "days_to_event": (row["date"] - event_date).days,
        })

    return pd.DataFrame(rows)


In [ ]:
# Pick an episode to inspect -- e.g. the one with the highest peak discharge
chosen = red_episodes.sort_values("peak_discharge", ascending=False).iloc[0]
case_study_date = chosen["start"]
print("Inspecting episode starting:", case_study_date, " peak discharge:", chosen["peak_discharge"])
    
case_result = backtest_event(feat_df, xgb_model, monthly_thresholds, case_study_date,
                              days_before=5, days_after=5)
case_result


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
color_map = {"GREEN": "#2ecc71", "ORANGE": "#e67e22", "RED": "#e74c3c"}

ax1.plot(pd.to_datetime(case_result["date"]), case_result["discharge"], color="steelblue", marker="o")
for _, r in case_result.iterrows():
    ax1.axvspan(pd.Timestamp(r["date"]) - pd.Timedelta(hours=12),
                pd.Timestamp(r["date"]) + pd.Timedelta(hours=12),
                color=color_map[r["alert_color"]], alpha=0.15)

ax1.axvline(pd.Timestamp(case_study_date), color="black", linestyle="--", label="Episode start (scan-detected)")
ax1.set_ylabel("Discharge (m3/s)")
ax1.set_title(f"Scan-detected case study starting {case_study_date}")
ax1.legend()
plt.tight_layout()
plt.show()


Cross-reference `case_study_date` and the other entries in `red_episodes` against any flood
records you can find for the Manafwa catchment (news archives, district disaster reports, your
MWE contacts) — episodes that line up with a documented event are strong, independently-derived
case studies; episodes that don't are worth a quick sense-check on whether the thresholds need
retuning for that period (e.g. the 1988–1992 anomaly your thesis already flagged).


## 13. Validating against ReliefWeb-reported flood/landslide events


Below, the dates pulled from ReliefWeb situation reports for the Mt Elgon subregion
(Bududa, Manafwa, Mbale, Sironko, Bulambuli, Namisindwa, Kapchorwa) — the same broader
catchment area, since these districts share the same rainfall system and several reports don't
isolate Manafwa district specifically. Treat the ±3 day windows as approximate: report dates are
often a few days *after* the actual rainfall/landslide onset.

| Event | Approx. date | Source |
|---|---|---|
| Nametsi, Bududa landslide | ~2010-03-01 | ReliefWeb, Uganda: New landslide deaths rise to eight |
| Mbale/Sironko/Kabale landslides | ~2010-05-16 | ReliefWeb, same report |
| Manafwa District landslides (30 houses) | ~2018-05-16 | ReliefWeb, "Landslides bury 30 houses..." |
| Bududa/Mbale/Butaleja/Sironko floods | ~2019-06-15 | ReliefWeb, DREF Operation MDRUG042 |
| Mt Elgon subregion flooding (25,000 displaced) | ~2019-11-01 | ReliefWeb, Govt PDNA report |
| Mbale/Kapchorwa/Bulambuli/Bududa floods | 2022-07-30 | ReliefWeb, DREF MDRUG046 |
| Bulambuli, River Simu floods | 2024-11-27 | ReliefWeb, Uganda Floods disaster page |

⚠️ These dates are approximate and drawn from report/publication timing, not precise
hydrological onset.


In [ ]:
REAL_EVENTS = {
    "2010-03 Nametsi landslide": "2010-03-01",
    "2010-05 Mbale/Sironko landslides": "2010-05-16",
    "2018-05 Manafwa landslides": "2018-05-16",
    "2019-06 Bududa/Mbale floods": "2019-06-15",
    "2019-11 Mt Elgon flooding": "2019-11-01",
    "2022-07 Mbale/Kapchorwa/Bulambuli floods": "2022-07-30",
    "2024-11 Bulambuli/River Simu floods": "2024-11-27",
}

def check_event_against_pipeline(feat_df, model, thresholds, event_name, event_date, window_days=3):
    result = backtest_event(feat_df, model, thresholds, event_date, days_before=7, days_after=3)
    near = result[result["days_to_event"].abs() <= window_days]
    if near.empty:
        return {"event": event_name, "date": event_date, "status": "NO DATA IN RANGE"}
    best = near.loc[near["criteria_met"].idxmax()]
    return {
        "event": event_name,
        "date": event_date,
        "best_alert": best["alert_color"],
        "best_day": best["date"],
        "criteria_met": best["criteria_met"],
        "crit_model": best["crit_model"],
        "crit_threshold": best["crit_threshold"],
        "crit_persistence": best["crit_persistence"],
        "crit_onset": best["crit_onset(*)"],
    }

validation_rows = []
for name, d in REAL_EVENTS.items():
    try:
        validation_rows.append(check_event_against_pipeline(feat_df, xgb_model, monthly_thresholds, name, d))
    except ValueError as e:
        validation_rows.append({"event": name, "date": d, "status": f"SKIPPED: {e}"})

validation_df = pd.DataFrame(validation_rows)
validation_df
